# 🇱🇰 Piper TTS — Sinhala Voice Training

Train a high-quality Sinhala text-to-speech model using [Piper](https://github.com/rhasspy/piper) (VITS architecture) on Google Colab.

## ⚠️ Important Notes

- **GPU Required:** Go to Runtime → Change runtime type → **T4 GPU**
- **Training Time:** ~20–40 hours on T4. You'll need multiple Colab sessions.
- **Session Limits:** Free Colab disconnects after ~4–12h. Checkpoints save to Google Drive so you can resume.
- **Pro Tip:** Colab Pro gives longer sessions and priority GPU access — highly recommended for this.
- **Strategy:** Run training overnight / in background tabs. Resume from checkpoint each session.

## 1. Setup & Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive
!mkdir -p /content/drive/MyDrive/piper-sinhala/{checkpoints,exported}

In [ ]:
# Install eSpeak-ng (has built-in Sinhala G2P support)
!apt-get -qq install -y espeak-ng

# Verify Sinhala support
!espeak-ng --voices | grep si
!echo 'සිංහල පරීක්ෂණය' | espeak-ng -v si -x -q

In [ ]:
# Install piper-phonemize and training dependencies
!pip install -q piper-phonemize

# Clone piper training repo
!git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper 2>/dev/null || echo 'Already cloned'

# Install training requirements
%cd /content/piper/src/python
!pip install -q -e .
!pip install -q torchmetrics==0.11.4 lightning==2.0.9.post0

# Additional utilities
!pip install -q librosa soundfile matplotlib tqdm
!apt-get -qq install -y sox libsox-fmt-all ffmpeg

## 2. Download Sinhala Datasets

We use two OpenSLR datasets:
- **OpenSLR 30** — Google's high-quality Sinhala multi-speaker TTS corpus (primary)
- **OpenSLR 52** — Sinhala ASR data (supplementary, more diverse but noisier)

In [ ]:
!mkdir -p /content/datasets
%cd /content/datasets

# OpenSLR 30 — Sinhala multi-speaker TTS
!wget -q --show-progress -c https://www.openslr.org/resources/30/si_lk_female.zip
!wget -q --show-progress -c https://www.openslr.org/resources/30/si_lk_male.zip

# OpenSLR 52 — Sinhala ASR data
!wget -q --show-progress -c https://www.openslr.org/resources/52/asr_sinhala_0.zip

print("\n✅ Downloads complete")

In [ ]:
# Extract datasets
!unzip -qo si_lk_female.zip -d si_lk_female
!unzip -qo si_lk_male.zip -d si_lk_male
!unzip -qo asr_sinhala_0.zip -d asr_sinhala

# Inspect structure
!echo '=== OpenSLR 30 (female) ===' && ls si_lk_female/ && echo
!echo '=== OpenSLR 30 (male) ===' && ls si_lk_male/ && echo
!echo '=== OpenSLR 52 ===' && ls asr_sinhala/ && echo
!find . -name '*.txt' -o -name '*.tsv' -o -name '*.csv' | head -20

## 3. Data Preparation

Steps:
1. Convert all audio to 22050Hz 16-bit mono WAV
2. Clean & normalize Sinhala text
3. Create Piper-compatible metadata (`audio_path|text`)
4. Train/validation split (95/5)
5. Generate phoneme cache

In [ ]:
import os
import re
import csv
import glob
import random
import subprocess
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR = Path('/content/piper_dataset')
WAV_DIR = OUTPUT_DIR / 'wav'
WAV_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22050

def clean_sinhala_text(text):
    """Normalize Sinhala text for TTS training."""
    text = text.strip()
    # Remove non-Sinhala, non-punctuation characters (keep Sinhala Unicode block U+0D80-U+0DFF)
    text = re.sub(r'[^\u0D80-\u0DFF\s.,!?;:\-–—\'\"()0-9]', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def convert_audio(src, dst):
    """Convert audio to 22050Hz 16-bit mono WAV."""
    subprocess.run(
        ['ffmpeg', '-y', '-i', str(src), '-ar', str(SAMPLE_RATE),
         '-ac', '1', '-sample_fmt', 's16', str(dst)],
        capture_output=True
    )

print('Helper functions defined ✅')

In [ ]:
# Parse OpenSLR 30 datasets (both speakers)
# Structure: line_index.tsv maps filenames to text

metadata = []  # list of (wav_path, text)

def parse_openslr30(base_dir, speaker_tag):
    """Parse OpenSLR 30 directory structure."""
    entries = []
    base = Path(base_dir)
    # Look for transcript files
    for tsv_file in sorted(base.rglob('line_index.tsv')):
        audio_dir = tsv_file.parent
        with open(tsv_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    fname, text = parts[0], parts[-1]
                    # Find the audio file
                    audio_src = audio_dir / f"{fname}.wav"
                    if not audio_src.exists():
                        audio_src = audio_dir / f"{fname}.mp3"
                    if not audio_src.exists():
                        for ext in ['wav', 'mp3', 'flac']:
                            candidates = list(audio_dir.rglob(f"{fname}.{ext}"))
                            if candidates:
                                audio_src = candidates[0]
                                break
                    if audio_src.exists():
                        text = clean_sinhala_text(text)
                        if len(text) > 2:
                            out_name = f"{speaker_tag}_{fname}.wav"
                            entries.append((audio_src, WAV_DIR / out_name, text))
    return entries

entries_f = parse_openslr30('/content/datasets/si_lk_female', 'f')
entries_m = parse_openslr30('/content/datasets/si_lk_male', 'm')
print(f"OpenSLR 30 female: {len(entries_f)} utterances")
print(f"OpenSLR 30 male: {len(entries_m)} utterances")

# For single-speaker training, pick the speaker with more data
# (You can change this to train multi-speaker later)
all_entries = entries_f if len(entries_f) >= len(entries_m) else entries_m
print(f"\nUsing speaker with {len(all_entries)} utterances for single-speaker training")

# Optionally include OpenSLR 52 (uncomment if you want more data)
# Note: ASR data may be noisier, only use if you need more volume
# entries_52 = parse_openslr30('/content/datasets/asr_sinhala', 'asr')
# all_entries += entries_52

In [ ]:
# Convert audio files to target format
print(f"Converting {len(all_entries)} audio files to {SAMPLE_RATE}Hz mono WAV...")

final_metadata = []
for src, dst, text in tqdm(all_entries):
    convert_audio(src, dst)
    if dst.exists() and dst.stat().st_size > 1000:  # skip failed conversions
        final_metadata.append((dst.name, text))

print(f"\n✅ {len(final_metadata)} valid utterances prepared")

In [ ]:
# Train/validation split (95/5)
random.seed(42)
random.shuffle(final_metadata)

split_idx = int(len(final_metadata) * 0.95)
train_meta = final_metadata[:split_idx]
val_meta = final_metadata[split_idx:]

def write_metadata(entries, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        for wav_name, text in entries:
            # Piper format: audio_path|text (path relative to wav dir)
            f.write(f"{wav_name}|{text}\n")

write_metadata(train_meta, OUTPUT_DIR / 'metadata_train.csv')
write_metadata(val_meta, OUTPUT_DIR / 'metadata_val.csv')

print(f"Train: {len(train_meta)} utterances")
print(f"Val:   {len(val_meta)} utterances")
print(f"\nSample entries:")
for name, text in train_meta[:3]:
    print(f"  {name} | {text[:60]}...")

In [ ]:
# Generate phoneme cache using eSpeak-ng Sinhala
# This pre-computes phoneme sequences to speed up training

from piper_phonemize import phonemize_espeak

PHONEME_CACHE = OUTPUT_DIR / 'phoneme_cache'
PHONEME_CACHE.mkdir(exist_ok=True)

print("Generating phoneme cache...")
phoneme_map = {}

for wav_name, text in tqdm(final_metadata):
    try:
        phonemes = phonemize_espeak(text, 'si')  # Sinhala language code
        phoneme_str = ' '.join([' '.join(sent) for sent in phonemes])
        phoneme_map[wav_name] = phoneme_str
        
        # Write individual cache file
        cache_file = PHONEME_CACHE / f"{Path(wav_name).stem}.txt"
        cache_file.write_text(phoneme_str, encoding='utf-8')
    except Exception as e:
        print(f"  ⚠️ Failed: {wav_name}: {e}")

print(f"\n✅ Cached phonemes for {len(phoneme_map)} utterances")
# Show a sample
sample_key = list(phoneme_map.keys())[0]
print(f"\nSample: {sample_key}")
print(f"  Phonemes: {phoneme_map[sample_key][:100]}...")

## 4. Configure Training

We use Piper's VITS configuration for single-speaker training.

Key settings:
- **batch_size=32** — fits T4 16GB VRAM
- **learning_rate=2e-4** — standard for VITS
- **epochs=1000** — ~20-40h on T4 depending on dataset size
- **Checkpoints every 1000 steps** — saved to Google Drive

In [ ]:
import json

CHECKPOINT_DIR = Path('/content/drive/MyDrive/piper-sinhala/checkpoints')

# Piper VITS training config
training_config = {
    "audio": {
        "sample_rate": 22050,
        "mel_channels": 80,
        "hop_length": 256,
        "win_length": 1024,
        "fft_size": 1024,
        "fmin": 0,
        "fmax": None
    },
    "model": {
        "inter_channels": 192,
        "hidden_channels": 192,
        "filter_channels": 768,
        "n_heads": 2,
        "n_layers": 6,
        "kernel_size": 3,
        "p_dropout": 0.1,
        "resblock": "1",
        "resblock_kernel_sizes": [3, 7, 11],
        "resblock_dilation_sizes": [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        "upsample_rates": [8, 8, 2, 2],
        "upsample_initial_channel": 512,
        "upsample_kernel_sizes": [16, 16, 4, 4],
        "n_speakers": 0,
        "gin_channels": 0
    },
    "training": {
        "seed": 1234,
        "epochs": 1000,
        "learning_rate": 2e-4,
        "betas": [0.8, 0.99],
        "eps": 1e-9,
        "batch_size": 32,
        "fp16_run": True,
        "lr_decay": 0.999875,
        "segment_size": 8192,
        "init_lr_ratio": 1,
        "warmup_epochs": 0,
        "c_mel": 45,
        "c_kl": 1.0
    },
    "data": {
        "training_files": str(OUTPUT_DIR / 'metadata_train.csv'),
        "validation_files": str(OUTPUT_DIR / 'metadata_val.csv'),
        "wav_dir": str(WAV_DIR),
        "phoneme_cache_dir": str(PHONEME_CACHE),
        "max_wav_value": 32768.0,
        "filter_length": 1024,
        "hop_length": 256,
        "win_length": 1024,
        "mel_fmin": 0.0,
        "mel_fmax": None
    },
    "espeak": {
        "voice": "si"
    },
    "phoneme_type": "espeak",
    "language": {
        "code": "si",
        "name_english": "Sinhala",
        "name_native": "සිංහල"
    },
    "checkpoint": {
        "dir": str(CHECKPOINT_DIR),
        "save_every_n_steps": 1000
    }
}

config_path = OUTPUT_DIR / 'config.json'
with open(config_path, 'w') as f:
    json.dump(training_config, f, indent=2)

# Also save a copy to Drive
import shutil
shutil.copy(config_path, CHECKPOINT_DIR / 'config.json')

print(f"✅ Config saved to {config_path}")
print(f"✅ Config backed up to {CHECKPOINT_DIR / 'config.json'}")
print(f"\nKey settings:")
print(f"  Batch size:     {training_config['training']['batch_size']}")
print(f"  Learning rate:  {training_config['training']['learning_rate']}")
print(f"  Epochs:         {training_config['training']['epochs']}")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")

## 5. Train

### Handling Colab Disconnects

Training saves checkpoints to Google Drive every 1000 steps. When Colab disconnects:
1. Reconnect and re-run sections 1 (Setup) and 3–4 (to rebuild paths)
2. The training cell below auto-detects the latest checkpoint and resumes

**Expected timeline on T4:**
- ~20–40 hours total depending on dataset size
- Each Colab session: ~4–12 hours
- You'll need 3–8 sessions to complete training

In [ ]:
# Find latest checkpoint for resume
import glob

def find_latest_checkpoint(checkpoint_dir):
    """Find the latest checkpoint in the directory."""
    ckpts = sorted(glob.glob(str(Path(checkpoint_dir) / '*.ckpt')))
    if ckpts:
        latest = ckpts[-1]
        print(f"🔄 Found checkpoint: {latest}")
        return latest
    print("🆕 No checkpoint found — starting fresh")
    return None

resume_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)

In [ ]:
# Launch training
%cd /content/piper/src/python

resume_flag = f"--resume_from_checkpoint {resume_ckpt}" if resume_ckpt else ""

!python -m piper_train \
    --dataset-dir {OUTPUT_DIR} \
    --accelerator gpu \
    --devices 1 \
    --batch-size {training_config['training']['batch_size']} \
    --validation-split 0.05 \
    --num-test-examples 3 \
    --max_epochs {training_config['training']['epochs']} \
    --checkpoint-epochs 1 \
    --precision 16 \
    --quality medium \
    --default_root_dir {CHECKPOINT_DIR} \
    {resume_flag}

In [ ]:
# Display loss curves (run this in a separate cell while training)
import matplotlib.pyplot as plt
from pathlib import Path

# Parse training logs for loss values
log_dir = CHECKPOINT_DIR / 'lightning_logs'
if log_dir.exists():
    from tensorboard.backend.event_processing import event_accumulator
    
    versions = sorted(log_dir.glob('version_*'))
    if versions:
        latest_version = versions[-1]
        event_files = list(latest_version.glob('events.out.tfevents.*'))
        if event_files:
            ea = event_accumulator.EventAccumulator(str(latest_version))
            ea.Reload()
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            for i, tag in enumerate(['loss/g/total', 'loss/d/total']):
                if tag in ea.Tags()['scalars']:
                    events = ea.Scalars(tag)
                    steps = [e.step for e in events]
                    values = [e.value for e in events]
                    axes[i].plot(steps, values, alpha=0.7)
                    axes[i].set_title(tag)
                    axes[i].set_xlabel('Step')
                    axes[i].set_ylabel('Loss')
                    axes[i].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        else:
            print('No event files found yet. Training may still be starting.')
    else:
        print('No training versions found yet.')
else:
    print('Log directory not found. Start training first.')
    print('\nAlternatively, launch TensorBoard:')
    print(f'  %load_ext tensorboard')
    print(f'  %tensorboard --logdir {CHECKPOINT_DIR}')

## 6. Export & Test

After training completes (or at a good checkpoint), export the model to ONNX for fast CPU inference.

In [ ]:
# Export to ONNX
EXPORT_DIR = Path('/content/drive/MyDrive/piper-sinhala/exported')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Find the best/latest checkpoint
latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if latest_ckpt is None:
    print("❌ No checkpoint found! Train first.")
else:
    %cd /content/piper/src/python
    
    !python -m piper_train.export_onnx \
        {latest_ckpt} \
        {EXPORT_DIR / 'si-sinhala-medium.onnx'}
    
    # Copy config alongside model (required by piper)
    import json, shutil
    
    model_config = {
        "language": {
            "code": "si",
            "family": "si",
            "region": "LK",
            "name_english": "Sinhala",
            "name_native": "සිංහල"
        },
        "espeak": {
            "voice": "si"
        },
        "audio": {
            "sample_rate": 22050,
            "quality": "medium"
        },
        "num_speakers": 1,
        "speaker_id_map": {}
    }
    
    with open(EXPORT_DIR / 'si-sinhala-medium.onnx.json', 'w') as f:
        json.dump(model_config, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Model exported to {EXPORT_DIR}")
    !ls -lh {EXPORT_DIR}/

In [ ]:
# Test the exported model with sample Sinhala text
!pip install -q piper-tts

from piper import PiperVoice
import wave
import numpy as np
from IPython.display import Audio, display

model_path = str(EXPORT_DIR / 'si-sinhala-medium.onnx')
config_path = str(EXPORT_DIR / 'si-sinhala-medium.onnx.json')

voice = PiperVoice.load(model_path, config_path=config_path)

# Test sentences
test_sentences = [
    "ආයුබෝවන්, මම සිංහලෙන් කතා කරමි.",
    "අද කාලගුණය ඉතාම සුන්දරයි.",
    "ශ්‍රී ලංකාව ඉතාම ලස්සන රටකි.",
]

for i, text in enumerate(test_sentences):
    out_path = f"/content/test_output_{i}.wav"
    with wave.open(out_path, 'w') as wav_file:
        voice.synthesize(text, wav_file)
    
    print(f"\n📝 '{text}'")
    display(Audio(out_path, autoplay=False))

## 7. Download Model

Package the trained model for deployment.

In [ ]:
# Package model for deployment
import shutil

PACKAGE_DIR = Path('/content/piper-si-sinhala-medium')
PACKAGE_DIR.mkdir(exist_ok=True)

shutil.copy(EXPORT_DIR / 'si-sinhala-medium.onnx', PACKAGE_DIR)
shutil.copy(EXPORT_DIR / 'si-sinhala-medium.onnx.json', PACKAGE_DIR)

# Create a README
readme = """# Piper TTS — Sinhala (si_LK) Medium

Single-speaker Sinhala TTS model trained with Piper (VITS).

## Usage

```bash
# Install piper
pip install piper-tts

# Synthesize speech
echo 'ආයුබෝවන්' | piper \\
  --model si-sinhala-medium.onnx \\
  --config si-sinhala-medium.onnx.json \\
  --output_file output.wav

# Or use the Python API
from piper import PiperVoice
import wave

voice = PiperVoice.load('si-sinhala-medium.onnx', config_path='si-sinhala-medium.onnx.json')
with wave.open('output.wav', 'w') as f:
    voice.synthesize('ආයුබෝවන්, මම සිංහලෙන් කතා කරමි.', f)
```

## Details
- Architecture: VITS (medium quality)
- Sample rate: 22050 Hz
- Language: Sinhala (si)
- Phonemizer: eSpeak-ng
- Training data: OpenSLR 30
"""

(PACKAGE_DIR / 'README.md').write_text(readme)

# Create tar.gz archive
!cd /content && tar czf piper-si-sinhala-medium.tar.gz piper-si-sinhala-medium/
# Copy to Drive
shutil.copy('/content/piper-si-sinhala-medium.tar.gz', '/content/drive/MyDrive/piper-sinhala/')

print("✅ Model packaged!")
!ls -lh /content/piper-si-sinhala-medium.tar.gz

# Download link in Colab
from google.colab import files
files.download('/content/piper-si-sinhala-medium.tar.gz')

---

## 📋 Quick Reference — Resume After Disconnect

When Colab disconnects mid-training:

1. **Reconnect** and select GPU runtime
2. **Run Section 1** (Setup & Dependencies) — reinstalls packages
3. **Run Section 4** (Configure Training) — rebuilds config with paths
4. **Run Section 5** (Train) — auto-detects latest checkpoint from Drive and resumes

No need to re-download or re-prepare data if it's still in `/content/` (usually cleared on disconnect though).
If data is gone, also re-run Sections 2–3.

### Monitoring Progress
- Watch the training cell output for loss values
- Run the loss curves cell periodically
- Use TensorBoard: `%tensorboard --logdir /content/drive/MyDrive/piper-sinhala/checkpoints`
- Good convergence: generator loss should steadily decrease over time
- Listen to test samples every ~5000 steps to check quality